# Linear elasticity: material robustness

2D cantilevered rectangular beam under gravity. Fixed on the left face; traction-free on the right, top, and bottom.

Hellinger–Reissner mixed formulation with the Johnson–Mercier element (symmetric $P_1$ stress in $H(\operatorname{div})$, $P_0$ displacement). The zero-displacement BC on the clamped face is natural (prescribed displacement enters the RHS and vanishes since $g=0$); zero traction on the free faces is the essential BC on the stress space.

In [ ]:
try:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-development-real.sh" -O "/tmp/firedrake-install.sh"
    !bash "/tmp/firedrake-install.sh"
    from firedrake import *  # noqa: F401
except:
    from firedrake import *  # noqa: F401

In [ ]:
# Rectangular beam: length x height
length, height = 10.0, 1.0
mesh = RectangleMesh(50, 5, length, height)

# Material parameters
E   = Constant(1e6)   # Young's modulus
nu  = Constant(0.3)   # Poisson's ratio
rho = Constant(1e3)   # density
g   = Constant(9.81)  # gravitational acceleration

mu    = E / (2*(1 + nu))
lmbda = E*nu / ((1 + nu)*(1 - 2*nu))

In [ ]:
# JM element: symmetric P1 stress in H(div), P0 displacement
Sigma = FunctionSpace(mesh, "JM", 1)
V     = VectorFunctionSpace(mesh, "DG", 0)
W     = Sigma * V

sigma, u = TrialFunctions(W)
tau,   v = TestFunctions(W)

d = 2  # spatial dimension

def A(s):
    """Compliance tensor: inverse of the Lame elasticity tensor."""
    return (1/(2*mu))*s - (lmbda/(2*mu*(2*mu + d*lmbda)))*tr(s)*Identity(d)

f   = as_vector([0, -rho*g])  # gravity
n   = FacetNormal(mesh)
g_D = Constant((0, 0))        # clamped: zero displacement on left face

# Hellinger-Reissner saddle-point form
a = (inner(A(sigma), tau) + inner(u, div(tau)) + inner(div(sigma), v)) * dx
L = - inner(f, v) * dx + inner(g_D, dot(tau, n)) * ds(1)  # weak: u = g_D on Gamma_D

# Essential BC: zero traction on free faces (right=2, bottom=3, top=4)
bc = DirichletBC(W.sub(0), Constant(((0, 0), (0, 0))), [2, 3, 4])

w_h = Function(W)
solve(a == L, w_h, bcs=bc, solver_parameters={"ksp_type": "preonly", "pc_type": "lu"})

sigma_h, u_h = w_h.subfunctions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

# Vertical displacement
c0 = tripcolor(u_h.sub(1), axes=axes[0], cmap="RdBu_r")
plt.colorbar(c0, ax=axes[0], label=r"$u_y$")
axes[0].set_aspect("equal")
axes[0].set_title("Vertical displacement")

# Von Mises stress
sig_vm = Function(FunctionSpace(mesh, "DG", 0), name="Von Mises stress")
sig_vm.interpolate(sqrt(
    sigma_h[0, 0]**2 - sigma_h[0, 0]*sigma_h[1, 1] + sigma_h[1, 1]**2
    + 3*sigma_h[0, 1]**2
))
c1 = tripcolor(sig_vm, axes=axes[1], cmap="inferno")
plt.colorbar(c1, ax=axes[1], label=r"$\sigma_\mathrm{vM}$")
axes[1].set_aspect("equal")
axes[1].set_title("Von Mises stress")

plt.tight_layout()
plt.show()